<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building RAG Agents with LLMs</b></font></h1>
<h2><b>Notebook 5: </b>대용량 문서 다루기</h2>
<br>

이전 노트북에서는 running state chain과 지식 베이스에 대해 배웠습니다! 노트북을 마칠 무렵에는 간단한 대화 관리와 커스텀 지식 추적에 필요한 도구를 모두 갖추게 되었습니다. 이 노트북에서는 같은 아이디어를 대용량 문서의 영역으로 옮겨, 큰 파일을 LLM 컨텍스트에 통합하려 할 때 어떤 문제에 부딪히게 되는지 살펴봅니다.

<br>

### **학습 목표:**

- 문서 로더(document loader)와 이들이 제공하는 유틸리티에 익숙해집니다.
- 문서를 청크(chunk)로 나누고 지식 베이스를 점진적으로 쌓아 가며, 제한된 컨텍스트 공간에서 대용량 문서를 파싱하는 방법을 배웁니다.
- 문서 청크의 점진적 재맥락화, 강제 변환, 통합이 얼마나 유용할 수 있는지, 그리고 어디에서 자연스러운 한계에 부딪히는지 이해합니다.

<br>

### **생각해 볼 질문:**

- ArxivParser에서 나온 청크들을 보면, 일부 청크는 그 자체로는 거의 의미가 없거나 텍스트 변환 과정에서 완전히 망가져 있음을 알 수 있습니다. 청크를 정리하기 위한 별도의 처리를 하고 있나요?
- 문서 요약 워크플로(또는 많은 문서 청크 목록을 처리하는 유사한 워크플로)를 고려할 때, 이 작업은 얼마나 자주 수행되어야 하며 언제 정당화될 수 있을까요?

<br>

### **환경 설정:**

In [ ]:
## Necessary for Colab, not necessary for course environment
# %pip install -qq langchain langchain-nvidia-ai-endpoints gradio
# %pip install -qq arxiv pymupdf
# !mkdir -p cached_papers && test -s cached_papers/2210.03629v3.pdf || wget -q --tries=3 --timeout=20 -O cached_papers/2210.03629v3.pdf https://arxiv.org/pdf/2210.03629v3

# import os
# os.environ["NVIDIA_API_KEY"] = "nvapi-..."

from functools import partial
from rich.console import Console
from rich.style import Style
from rich.theme import Theme

console = Console()
base_style = Style(color="#76B900", bold=True)
pprint = partial(console.print, style=base_style)

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
[m.id for m in ChatNVIDIA.get_available_models() if "nvidia" in m.id] 

In [ ]:
## Useful utility method for printing intermediate states
from langchain_core.runnables import RunnableLambda
from functools import partial

def RPrint(preface="State: "):
    def print_and_return(x, preface=""):
        print(f"{preface}{x}")
        return x
    return RunnableLambda(partial(print_and_return, preface=preface))

def PPrint(preface="State: "):
    def print_and_return(x, preface=""):
        pprint(preface, x)
        return x
    return RunnableLambda(partial(print_and_return, preface=preface))

----

<br>

## **Part 1:** 문서와 대화하기

이 노트북은 LLM을 사용해 문서와 대화하는 것에 관한 긴 논의의 시작입니다. 채팅 모델이 거대한 공개 데이터 저장소로 학습되고 커스텀 데이터로 재학습하는 것은 엄두를 못 낼 만큼 비싼 세상에서, LLM이 PDF 모음이나 심지어 YouTube 영상에 대해 추론하게 한다는 아이디어는 많은 기회를 열어 줍니다!

- **LLM이 사람이 읽을 수 있는 문서에 기반한, 수정 가능한 지식 베이스를 가질 수 있습니다.** 즉, 어떤 데이터에 접근할 수 있는지 직접 제어하고 그 데이터와 상호작용하도록 지시할 수 있습니다.

- **LLM이 문서 집합을 살펴보고 그로부터 직접 참조를 가져올 수 있습니다.** 충분한 프롬프트 엔지니어링과 지시 따르기 사전 성향이 있다면, 모델이 여러분이 제공한 자료에만 근거해 동작하도록 강제할 수 있습니다.

- **LLM이 문서와 상호작용하며 필요에 따라 자동으로 수정할 수도 있습니다.** 이는 자동 콘텐츠 정제와 합성 작업의 길을 열어 주며, 뒤에서 더 탐구합니다.

가능성을 나열하는 것은 쉽고, 거기서부터 상상력을 마음껏 펼칠 수 있습니다... 하지만 아직 이를 실현할 도구를 갖추지는 못했죠?

<br>

#### **단순한 접근: 문서 채워 넣기(Stuffing)**

텍스트 문서(PDF, 블로그 등)가 있고 그 내용과 관련된 질문을 하고 싶다고 합시다. 시도해 볼 수 있는 한 가지 접근법은 문서의 표현을 통째로 채팅 모델에 넣는 것입니다! 문서 관점에서 이는 [**document stuffing**](https://js.langchain.com/v0.1/docs/modules/chains/document/stuff/)이라고 합니다.

<!-- > <img src="https://drive.google.com/uc?export=view&id=14DRI_uDviqzqg14TKoIc8IlBc3Zsb8oO" width=800px/> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/doc_stuff.png" width=800px/>
>
> 출처: [**Stuff | LangChain**🦜️🔗](https://js.langchain.com/v0.1/docs/modules/chains/document/stuff/)

<br>

모델이 충분히 강력하고 문서가 충분히 짧다면 이 방법도 잘 동작할 수 있지만, 문서 전체에 대해 잘 동작하리라 기대해서는 안 됩니다. 많은 최신 LLM은 학습 상의 제약으로 인해 긴 컨텍스트를 다루는 데 상당한 어려움을 겪습니다. 요즘은 대규모 모델의 성능 저하가 그렇게 치명적이지는 않지만, (원시 모델에 접근한다고 가정할 때) 어떤 모델을 사용하든 좋은 지시 따르기 성능은 꽤 빠르게 무너질 가능성이 큽니다.

<br>

**문서 추론에서 해결해야 할 핵심 문제는 다음과 같습니다:**

- 문서를 추론 가능한 조각으로 어떻게 나눌 것인가?

- 문서의 크기와 수가 늘어날 때 이 조각들을 어떻게 효율적으로 찾고 고려할 것인가?

이 코스에서는 LLM 오케스트레이션 역량을 계속 쌓아 가면서 이 문제들을 해결하기 위한 여러 접근법을 탐구합니다. ***이 노트북은 이전의 running chain 역량을 더 점진적인 추론 방식으로 확장하는 역할을 하며, 다음 노트북들에서는 대규모 검색을 제대로 다루기 위한 새로운 기법들을 소개합니다.*** 이 과정에서 최신 오픈소스 솔루션을 계속 활용하여 우리의 솔루션이 표준적이고 통합 가능하도록 만들 것입니다.

말이 나온 김에, 문서 로딩 프레임워크 분야에는 강력한 선택지가 많으며, 코스 전반에서 두 주요 주자가 등장합니다:

- [**LangChain**](https://python.langchain.com/docs/get_started/introduction)은 일반적인 청킹 전략과 임베딩 프레임워크/서비스와의 강력한 통합을 통해 LLM을 자체 데이터 소스에 연결하는 간단한 프레임워크를 제공합니다. 이 프레임워크는 처음에 LLM 기능에 대한 폭넓은 지원을 중심으로 성장했으며, 이는 체인 추상화와 에이전트 조정 쪽에 강점이 있음을 시사합니다.

- [**LlamaIndex**](https://gpt-index.readthedocs.io/en/stable/)는 LLM 애플리케이션이 비공개 또는 도메인 특화 데이터를 수집하고, 구조화하고, 접근하기 위한 데이터 프레임워크입니다. 이후 LangChain과 유사한 일반 LLM 기능도 포함하도록 확장되었지만, 초기 추상화가 문서 문제를 중심으로 만들어졌기 때문에 현재까지도 LLM 구성 요소의 문서 측면을 다루는 데 가장 강합니다.

LlamaIndex와 LangChain 각각의 고유한 강점을 더 읽어 보고 자신에게 가장 잘 맞는 것을 고르는 것을 권장합니다. LlamaIndex는 LangChain과 *함께* 사용할 수 있으므로, 두 프레임워크의 고유한 기능을 큰 문제 없이 함께 활용할 수 있습니다. 단순함을 위해 이 코스에서는 LangChain을 고수하며, 관심 있는 분들을 위한 더 깊은 LlamaIndex 옵션은 [**NVIDIA/GenerativeAIExamples 저장소**](https://github.com/NVIDIA/GenerativeAIExamples/tree/main/RAG/notebooks)에 맡기겠습니다.

----

<br>

## **Part 2:** 문서 로딩

LangChain은 다양한 소스와 위치(로컬 저장소, 비공개 S3 버킷, 공개 웹사이트, 메시징 API 등)에서 다양한 문서 형식(HTML, PDF, 코드)을 수집할 수 있도록 여러 [문서 로더](https://docs.langchain.com/oss/python/integrations/document_loaders)를 제공합니다. 이 로더들은 데이터 소스를 조회하여 콘텐츠와 메타데이터를 담은 `Document` 객체를 반환하며, 보통 평문 텍스트 또는 사람이 읽을 수 있는 형식입니다. 이미 만들어져 바로 사용할 수 있는 문서 로더가 많으며, LangChain 자체 제공 옵션은 [여기](https://docs.langchain.com/oss/python/integrations/document_loaders)에 나열되어 있습니다.

**이 예제에서는 다음 LangChain 로더 중 하나를 사용해 원하는 연구 논문을 로드할 수 있습니다:**
- [`UnstructuredFileLoader`](https://reference.langchain.com/python/langchain-community/document_loaders/unstructured/UnstructuredFileLoader): 임의의 파일에 두루 쓸 수 있는 파일 로더로, 문서 구조에 대해 많은 가정을 하지 않으며 대체로 충분합니다.
- [`ArxivLoader`](https://reference.langchain.com/python/langchain-community/document_loaders/arxiv/ArxivLoader): Arxiv 인터페이스와 직접 통신할 수 있는 더 특화된 파일 로더입니다. [많은 로더 중 하나의 예](https://docs.langchain.com/oss/python/integrations/document_loaders)일 뿐이지만, 데이터에 대해 더 많은 가정을 하여 더 깔끔한 파싱과 메타데이터 자동 채우기를 제공합니다(여러 문서/형식이 있을 때 유용합니다).

코드 예제에서는 기본적으로 `ArxivLoader`를 사용해 [MRKL](https://arxiv.org/abs/2205.00445) 또는 [ReAct](https://arxiv.org/abs/2210.03629) 논문 중 하나를 로드합니다. 채팅 모델 연구를 계속하다 보면 언젠가 마주칠 가능성이 높은 논문들입니다.

In [ ]:
%%time
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_community.document_loaders import ArxivLoader, PyMuPDFLoader
from pathlib import Path

## Loading in the file

## Unstructured File Loader: Good for arbitrary "probably good enough" loader
# documents = UnstructuredFileLoader("llama2_paper.pdf").load()

## More specialized loader, won't work for everything, but simple API and usually better results
try:
    documents = ArxivLoader(query="2210.03629").load()  ## ReAct
except Exception as error:
    cached_paper = Path("cached_papers/2210.03629v3.pdf")
    if not cached_paper.is_file():
        raise RuntimeError("Live arXiv retrieval failed and the cached ReAct paper is missing.") from error
    print("Live arXiv retrieval was unavailable; loading the cached ReAct paper.")
    documents = PyMuPDFLoader(str(cached_paper)).load()
# documents = ArxivLoader(query="2404.03622").load()  ## Visualization-of-Thought
# documents = ArxivLoader(query="2404.19756").load()  ## KAN: Kolmogorov-Arnold Networks
# documents = ArxivLoader(query="2404.07143").load()  ## Infini-Attention
# documents = ArxivLoader(query="2210.03629").load()  ## ReAct

<br>

임포트 결과에서 이 커넥터가 두 가지 구성 요소에 접근할 수 있게 해 준다는 것을 알 수 있습니다:
- `page_content`는 사람이 해석할 수 있는 형식의 실제 문서 본문입니다.
- `metadata`는 커넥터가 데이터 소스를 통해 제공하는 문서에 관한 관련 정보입니다.

아래에서 문서 본문의 길이를 확인해 안에 무엇이 있는지 볼 수 있으며, 아마 감당하기 어려운 문서 길이를 보게 될 것입니다:

In [ ]:
## Printing out a sample of the content
print("Number of Documents Retrieved:", len(documents))
print(f"Sample of Document 1 Content (Total Length: {len(documents[0].page_content)}):")
print(documents[0].page_content[:1000])

<br>

반면 메타데이터는 훨씬 보수적인 크기로, 여러분이 즐겨 쓰는 채팅 모델의 컨텍스트 구성 요소로 쓸 수 있을 정도입니다:

In [ ]:
pprint(documents[0].metadata)

<br>

메타데이터 형식을 그대로 받아들이고 본문은 완전히 무시하고 싶을 수도 있지만, 전문을 파고들지 않고는 접근할 수 없는 핵심 기능들이 있습니다:

- **메타데이터는 보장되지 않습니다.** `arxiv`의 경우 논문 초록, 제목, 저자, 날짜는 제출의 필수 구성 요소이므로 조회할 수 있는 것이 놀랍지 않습니다. 하지만 임의의 PDF나 웹페이지라면 반드시 그렇지는 않습니다.
- **에이전트가 문서 내용 깊숙이 들어갈 수 없습니다.** 요약은 알아 두면 좋고 그대로 사용할 수 있지만, 본문과 어떤 식으로든 상호작용하는 직접적인 경로를 제공하지는 않습니다(적어도 지금까지 배운 것으로는).
- **에이전트가 여전히 너무 많은 문서를 한 번에 추론할 수 없습니다.** MRKL/ReAct 예제에서는 두 요약을 하나의 컨텍스트로 합쳐 질문할 수 있을지도 모릅니다. 하지만 5개 문서와 한 번에 상호작용해야 한다면? 디렉터리 전체라면? 곧 관심 있는 문서를 요약하거나 나열하는 것만으로도 컨텍스트 윈도우가 정보로 과부하되는 것을 보게 될 것입니다!

----

<br>

## **Part 3:** 문서 변환하기

문서를 로드한 뒤 LLM에 컨텍스트로 전달하려면 변환이 필요한 경우가 많습니다. 변환 방법 중 하나가 **청킹(chunking)** 으로, 큰 콘텐츠 덩어리를 더 작은 세그먼트로 나눕니다. 이 기법은 [벡터 데이터베이스에서 반환되는 콘텐츠의 관련성을 최적화](https://www.pinecone.io/learn/chunking-strategies/)하는 데 도움이 되기 때문에 가치가 있습니다.

LangChain은 [다양한 문서 변환기](https://reference.langchain.com/python/langchain-community/document_transformers)를 제공하며, 그중 [``RecursiveCharacterTextSplitter``](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter)를 사용하겠습니다. 이 옵션은 자연스러운 끊김 지점에 대한 선호도를 기준으로 문서를 분할할 수 있게 해 줍니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", ";", ",", " ", ""],
)

## Some nice custom preprocessing
# documents[0].page_content = documents[0].page_content.replace(". .", "")
docs_split = text_splitter.split_documents(documents)

# def include_doc(doc):
#     ## Some chunks will be overburdened with useless numerical data, so we'll filter it out
#     string = doc.page_content
#     if len([l for l in string if l.isalpha()]) < (len(string)//2):
#         return False
#     return True

# docs_split = [doc for doc in docs_split if include_doc(doc)]
print(len(docs_split))

In [ ]:
for i in (0, 1, 2, 15, -1):
    pprint(f"[Document {i}]")
    print(docs_split[i].page_content)
    pprint("="*64)

<br>

우리의 청킹 접근법은 꽤 단순하지만, 애플리케이션에서 최소한 무언가를 동작시키는 것이 얼마나 쉬운지 보여 줍니다. 모델이 컨텍스트로 효과적으로 다룰 수 있도록 청크 크기를 작게 유지하려고 노력했지만, 이 모든 조각들에 대해서는 어떻게 추론할까요?

**임의의 문서 집합에 대해 이 접근법을 확장하고 최적화할 때 고려할 수 있는 옵션은 다음과 같습니다:**

- 논리적 구분점이나 합성 기법을 파악하기(수동, 자동, LLM 보조 등).
- 고유하고 관련성 높은 정보가 풍부한 청크를 구성하고, 데이터베이스 효용을 극대화하기 위해 중복을 피하기.
- 문서의 성격에 맞게 청킹을 커스터마이징하여 청크가 맥락상 관련 있고 응집력 있게 만들기.
- 데이터베이스에서의 검색성과 관련성을 높이기 위해 각 청크에 핵심 개념, 키워드, 메타데이터 조각을 포함하기.
- 청킹 효과를 지속적으로 평가하고, 크기와 콘텐츠 풍부함 사이의 최적 균형을 위해 전략을 조정할 준비를 하기.
- 검색 시도를 개선하기 위해 (암묵적으로 생성되거나 명시적으로 지정된) 계층 시스템을 고려하기.
    - 관심이 있다면 [**LlamaIndex 인덱스 가이드의 트리 구조**](https://developers.llamaindex.ai/python/framework/module_guides/indexing/index_guide/)를 출발점으로 살펴보세요.

----

<br>

## **Part 4: [실습]** 요약 정제하기

대용량 문서를 자동으로 추론하기 위한 한 가지 아이디어는 LLM으로 밀도 높은 요약이나 지식 베이스를 만드는 것입니다. 이전 노트북에서 슬롯 채우기로 대화의 running history를 유지했던 것처럼, 문서 전체의 running history를 유지하는 데 문제가 있을까요?

이 섹션에서는 LLM의 흥미로운 응용인 **데이터의 대량 자동 정제, 강제 변환, 통합**에 집중합니다. 구체적으로 while 루프와 running state chain 방식을 사용해 문서 청크 집합을 요약하는 단순하지만 유용한 Runnable을 구현합니다. 이 과정은 흔히 [**"문서 정제(document refinement)"**](https://reference.langchain.com/python/langchain-classic/chains/combine_documents/refine/)라고 하며, 이전의 대화 중심 슬롯 채우기 실습과 대체로 비슷합니다. 유일한 차이는 이제 늘어나는 채팅 기록 대신 큰 문서를 다룬다는 점입니다.

<!-- > <img src="https://drive.google.com/uc?export=view&id=1J2XR8Cc8YSkVJMiJCknMkgA02mBT8riZ" width=1000px/> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/doc_refine.png" width=1000px/>
>
> 출처: [**Refine | LangChain**🦜️🔗](https://reference.langchain.com/python/langchain-classic/chains/combine_documents/refine/)

<br>

#### **DocumentSummaryBase 모델**

이전 노트북의 `KnowledgeBase` 클래스와 마찬가지로, 문서의 핵심을 담도록 설계된 `DocumentSummaryBase` 구조를 만들 수 있습니다. 아래 구조는 `running_summary` 필드로 모델에 최종 요약을 요청하면서, `main_ideas`와 `loose_ends` 필드를 병목으로 사용해 running summary가 너무 빠르게 변하지 않도록 시도합니다. 이는 프롬프트 엔지니어링으로 강제해야 하는 부분이므로, 이 정보가 어떻게 사용되는지 보여 주는 `summary_prompt`도 함께 제공됩니다. 선택한 모델에 맞게 필요에 따라 자유롭게 수정하세요.

In [ ]:
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.passthrough import RunnableAssign
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser

from langchain_nvidia_ai_endpoints import ChatNVIDIA

from pydantic import BaseModel, Field
from typing import List
from IPython.display import clear_output


class DocumentSummaryBase(BaseModel):
    running_summary: str = Field("", description="Running description of the document. Do not override; only update!")
    main_ideas: List[str] = Field([], description="Most important information from the document (max 3)")
    loose_ends: List[str] = Field([], description="Open questions that would be good to incorporate into summary, but that are yet unknown (max 3)")


summary_prompt = ChatPromptTemplate.from_template(
    "You are generating a running summary of the document. Make it readable by a technical user."
    " After this, the old knowledge base will be replaced by the new one. Make sure a reader can still understand everything."
    " Keep it short, but as dense and useful as possible! The information should flow from chunk to (loose ends or main ideas) to running_summary."
    " The updated knowledge base keep all of the information from running_summary here: {info_base}."
    "\n\n{format_instructions}. Follow the format precisely, including quotations and commas"
    "\n\nWithout losing any of the info, update the knowledge base with the following: {input}"
)

<br>

이 기회에 이전 노트북의 `RExtract` 함수도 다시 가져오겠습니다:

In [ ]:
def RExtract(pydantic_class, llm, prompt):
    '''
    Runnable Extraction module
    Returns a knowledge dictionary populated by slot-filling extraction
    '''
    parser = PydanticOutputParser(pydantic_object=pydantic_class)
    instruct_merge = RunnableAssign({'format_instructions' : lambda x: parser.get_format_instructions()})
    def preparse(string):
        if '{' not in string: string = '{' + string
        if '}' not in string: string = string + '}'
        string = (string
            .replace("\\_", "_")
            .replace("\n", " ")
            .replace("\]", "]")
            .replace("\[", "[")
        )
        # print(string)  ## Good for diagnostics
        return string
    return instruct_merge | prompt | llm | preparse | parser


<br>

이를 염두에 두고, 다음 코드는 for 루프에서 running state chain을 호출하여 문서를 순회합니다! 필요한 유일한 수정은 `parse_chain` 구현으로, 지난 노트북의 적절히 구성된 `RExtract` 체인을 통해 상태를 전달해야 합니다. 이후 시스템은 문서의 running summary를 꽤 잘 유지할 것입니다(사용하는 모델에 따라 프롬프트를 약간 손봐야 할 수는 있습니다).

In [ ]:
latest_summary = ""

## TODO: Use the techniques from the previous notebook to complete the exercise
def RSummarizer(knowledge, llm, prompt, verbose=False):
    '''
    Exercise: Create a chain that summarizes
    '''
    ###########################################################################################
    ## START TODO:

    def summarize_docs(docs):        
        ## TODO: Initialize the parse_chain appropriately; should include an RExtract instance.
        ## HINT: You can get a class using the <object>.__class__ attribute...
        parse_chain = RunnableAssign({'info_base' : (lambda x: None)})
        ## TODO: Initialize a valid starting state. Should be similar to notebook 4
        state = {}

        global latest_summary  ## If your loop crashes, you can check out the latest_summary
        
        for i, doc in enumerate(docs):
            ## TODO: Update the state as appropriate using your parse_chain component

            assert 'info_base' in state 
            if verbose:
                print(f"Considered {i+1} documents")
                pprint(state['info_base'])
                latest_summary = state['info_base']
                clear_output(wait=True)

        return state['info_base']
        
    ## END TODO
    ###########################################################################################
    
    return RunnableLambda(summarize_docs)

instruct_model = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}}).bind(max_tokens=4096)
instruct_llm = instruct_model | StrOutputParser()

## Take the first 10 document chunks and accumulate a DocumentSummaryBase
summarizer = RSummarizer(DocumentSummaryBase(), instruct_llm, summary_prompt, verbose=True)
summary = summarizer.invoke(docs_split[:15])

In [ ]:
pprint(latest_summary)

----

<br>

## **Part 5:** 합성 데이터 처리

LLM을 이용한 문서 요약 탐구를 마무리하면서, 더 넓은 맥락과 잠재적 과제를 짚어 볼 필요가 있습니다. 간결하고 의미 있는 요약을 추출하는 실행 가능한 방법을 보여 주었지만, 이런 접근법이 왜 중요한지와 그에 수반되는 복잡성을 생각해 봅시다.

#### **정제의 일반성**

이 "점진적 요약" 기법은 초기 데이터와 원하는 출력 형식에 대해 거의 가정을 하지 않는 시작용 체인일 뿐이라는 점에 유의하세요. 같은 기법을 알려진 메타데이터, 능동적 가정, 다운스트림 목표를 염두에 두고 합성 정제물을 생성하도록 폭넓게 확장할 수 있습니다.

**다음과 같은 잠재적 응용을 생각해 보세요:**

1. **데이터 집계**: 문서 청크의 원시 데이터를 일관되고 유용한 요약으로 변환하는 구조 구축.
2. **분류 및 하위 주제 분석**: 청크에서 얻은 인사이트를 정의된 카테고리로 분류하고, 각 카테고리 안에서 떠오르는 하위 주제를 추적하는 시스템 구축.
3. **밀도 높은 정보 청크로 통합**: 이 구조들을 정제하여 인사이트를 압축된 세그먼트로 증류하고, 더 깊은 분석을 위해 직접 인용을 풍부하게 포함.

이러한 응용은 대화형 채팅 모델이 접근하고 탐색할 수 있는 **도메인 특화 지식 그래프**의 생성을 암시합니다. [**LangChain Knowledge Graphs**](https://docs.langchain.com/oss/python/integrations/retrievers/graph_rag) 같은 도구로 이를 자동 생성하는 유틸리티가 이미 존재합니다. 이런 구조를 구축하고 탐색하기 위한 계층 구조와 도구를 개발해야 할 수도 있지만, 활용 사례에 맞는 충분한 지식 그래프를 제대로 정제할 수 있다면 실행 가능한 옵션입니다! 더 큰 시스템과 벡터 유사도에 의존하는 고급 지식 그래프 구축 기법에 관심이 있다면 [**LangChain x Neo4j 글**](https://blog.langchain.dev/using-a-knowledge-graph-to-implement-a-devops-rag-application/)이 흥미로울 것입니다.

### **대규모 데이터 처리의 과제 다루기**

우리의 접근법은 흥미로운 가능성을 열어 주지만, 특히 대량의 데이터를 다룰 때는 과제도 있습니다:

- **일반적 전처리의 한계**: 요약은 비교적 간단하지만, 다양한 맥락에서 보편적으로 효과적인 계층 구조를 개발하는 것은 어렵습니다.

- **세분성과 탐색 비용**: 계층 구조에서 세밀한 세분성을 달성하려면 리소스가 많이 들 수 있으며, 상호작용당 관리 가능한 컨텍스트 크기를 유지하기 위해 정교한 통합이나 광범위한 분기가 필요합니다.

- **정확한 지시 수행에 대한 의존성**: 현재 도구로 이런 계층 구조를 탐색하려면 강력한 프롬프트 엔지니어링을 갖춘 고성능 지시 튜닝 모델에 크게 의존해야 합니다. 추론 지연과 인자 예측 오류의 위험이 상당할 수 있으므로, 이 작업에 LLM을 사용하는 것은 도전이 될 수 있습니다.

코스를 진행하면서 이러한 과제들이 이후 기법들로 어떻게 해결되는지 계속 주목하세요. 

-----

<br>

## **Part 6:** 마무리

이 노트북의 목표는 채팅 모델을 위한 대용량 문서 처리와 관련된 문제와 기법을 소개하는 것이었습니다. 다음 노트북에서는 매우 다른 장단점을 가진 보완적인 도구인 **임베딩 모델을 이용한 시맨틱 검색**을 살펴봅니다.

### <font color="#76b900">**수고하셨습니다!**</font>

### **다음 단계:**
1. **[선택]** 노트북 상단의 **"생각해 볼 질문" 섹션**을 다시 읽고 가능한 답을 생각해 보세요.
2. **[선택]** 이 노트북은 몇 가지 기본적인 문서 처리 체인을 포함하지만, 대체로 같은 직관 위에 구축되며 역시 매우 유용한 [Map Reduce](https://reference.langchain.com/python/langchain-classic/chains/mapreduce) 체인은 다루지 않습니다. 좋은 다음 단계이니 꼭 살펴보세요!

---

<div style="width: 55%%; background-color: white; margin-top: 50px;"><center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png" width="300" /></a></center></div>